# Reglas de Calidad del Dato — Entrega 1

```
================================================================================
ENTREGA 1 - REGLAS DE CALIDAD DEL DATO
================================================================================
TFM: Analisis multidimensional del estres estudiantil (BA & IA)

Formaliza como CODIGO las reglas de calidad detectadas en el EDA (Objetivo 1).
Cada regla sigue el patron pedido en el alcance de la entrega:
  "regla que valida el numero de registros que tienen el campo X a NULO /
   con formato incorrecto / fuera de rango"

Cada regla devuelve:
  - registros evaluados
  - registros que INCUMPLEN la regla (y %)
  - resultado: OK (0 incumplimientos) / AVISO / ERROR segun umbral
  - accion correctiva aplicada o recomendada

Al final se imprime un cuadro resumen de todas las reglas y se exporta a CSV.

USO EN WINDOWS: ajustar BASE a la carpeta local, p. ej.:
  BASE = r"C:\\Users\\nicop\\OneDrive\\Documentos\\Maestria\\INESDI & UNIE\\BA & IA\\TFM\\Propuesta"
================================================================================
```

---
**Cómo leer este notebook.** Cada *regla de calidad* es una validación automática sobre una columna:
cuenta cuántos registros la **incumplen** y devuelve un estado:

- **OK** — 0 incumplimientos
- **AVISO** — hay incumplimientos pero dentro del umbral tolerado; se documenta la acción correctiva
- **ERROR** — supera el umbral: hay que corregir antes de modelar

La función `regla()` centraliza esa lógica: recibe una *máscara booleana* (una condición fila a fila
donde `True` = registro que incumple), el total evaluado y la acción correctiva. Al final, el cuadro
resumen consolida las 49 reglas y se exporta a CSV.


**Configuración y función `regla()`.** Define rutas (entrada `BASE`, salida `OUT_CSV`) y la función central: recibe una condición fila a fila (`True` = incumple), cuenta incumplimientos, calcula el % y asigna estado OK/AVISO/ERROR según el umbral tolerado. Cada llamada a `regla()` queda registrada en la lista `resultados` para el cuadro final.

In [1]:
import os
import pandas as pd

# ── Configuracion global ──────────────────────────────────────────────────────
import os

BASE = r"C:\Users\nicop\OneDrive\Documentos\Maestria\INESDI & UNIE\BA & IA\TFM\Propuesta"
ALMC = r"C:\Users\nicop\OneDrive\Documentos\Maestria\INESDI & UNIE\BA & IA\TFM\Propuesta\Entrega\Objetivo1"

OUT_DIR = os.path.join(ALMC, "Reglas de Calidad")
os.makedirs(OUT_DIR, exist_ok=True)   # crea la carpeta si no existe
OUT_CSV = os.path.join(OUT_DIR, "resultado_reglas_calidad.csv")

resultados = []   # acumulador global de reglas

def regla(dataset, nombre, df_o_serie_mask, total, umbral_aviso=0.0,
          accion="—"):
    """Registra una regla de calidad.
    df_o_serie_mask: mascara booleana con True = registro que INCUMPLE.
    umbral_aviso: % de incumplimiento tolerado antes de marcar ERROR.
    """
    n_bad = int(df_o_serie_mask.sum())
    pct = n_bad / total * 100 if total else 0
    if n_bad == 0:
        estado = "OK"
    elif pct <= umbral_aviso:
        estado = "AVISO"
    else:
        estado = "ERROR"
    resultados.append({
        "Dataset": dataset, "Regla": nombre, "Evaluados": total,
        "Incumplen": n_bad, "% incumplen": round(pct, 2),
        "Estado": estado, "Accion": accion,
    })
    print(f"  [{estado:5s}] {nombre}: {n_bad}/{total} ({pct:.2f}%) -> {accion}")

print("=" * 90)
print("REGLAS DE CALIDAD DEL DATO - EJECUCION")
print("=" * 90)

REGLAS DE CALIDAD DEL DATO - EJECUCION


## BLOQUE 1

**Reglas del bloque.** Carga cada dataset y ejecuta sus validaciones: nulos, rangos válidos, duplicados/claves únicas. Cada `regla()` imprime su resultado al momento.

In [2]:
print("\n--- B1.1 Academic Stress (poushal02) ---")
df = pd.read_csv(os.path.join(BASE, "Bloque 1",
    "1 Student Academic Stress Real World Dataset",
    "academic Stress level - maintainance 1.csv"))
df.columns = [c.strip() for c in df.columns]
T = "Rate your academic stress index"
n = len(df)
regla("B1.1 AcademicStress", "R1: columnas sin espacios sobrantes en el nombre",
      pd.Series([c != c.strip() for c in df.columns]), df.shape[1],
      accion="strip() aplicado a los nombres al cargar")
regla("B1.1 AcademicStress", "R2: indice de estres nulo", df[T].isna(), n,
      accion="Excluir registro del analisis del target")
regla("B1.1 AcademicStress", "R3: indice de estres fuera de rango [1,5]",
      ~df[T].dropna().between(1, 5), df[T].notna().sum(),
      accion="Ninguna necesaria si 0")
regla("B1.1 AcademicStress", "R4: registros duplicados completos",
      df.duplicated(), n, accion="Eliminar duplicados si aparecen")

print("\n--- B1.2 Anxiety GAD-7 (petalme) ---")
df = pd.read_csv(os.path.join(BASE, "Bloque 1", "1 Student anxiety dataset",
                              "anxiety.csv"), encoding="latin1")
n = len(df)
regla("B1.2 Anxiety", "R1: GAD_T nulo", df["GAD_T"].isna(), n,
      accion="Excluir del analisis de ansiedad")
regla("B1.2 Anxiety", "R2: GAD_T fuera de rango [0,21]",
      ~df["GAD_T"].dropna().between(0, 21), df["GAD_T"].notna().sum(),
      accion="Revisar calculo de la escala si >0")
regla("B1.2 Anxiety", "R3: Edad fuera de rango plausible [13,90]",
      ~df["Age"].dropna().between(13, 90), df["Age"].notna().sum(),
      umbral_aviso=1.0, accion="Filtrar registros fuera de rango")
regla("B1.2 Anxiety", "R4: Horas de juego semanales imposibles (>112)",
      df["Hours"].dropna() > 112, df["Hours"].notna().sum(),
      umbral_aviso=1.0, accion="Tratar como atipico/captura erronea (winsorizar o excluir)")
regla("B1.2 Anxiety", "R5: Work nulo (impide filtrar estudiantes)",
      df["Work"].isna(), n, umbral_aviso=5.0,
      accion="Los nulos quedan fuera del subset estudiantil")

print("\n--- B1.3 Mental Health Survey (abdullahashfaqvirk) ---")
df = pd.read_csv(os.path.join(BASE, "Bloque 1", "1 Student Mental Health Survey",
                              "MentalHealthSurvey.csv"))
n = len(df)
for col in ["depression", "anxiety"]:
    regla("B1.3 MHSurvey", f"R: {col} nulo", df[col].isna(), n,
          accion="Excluir del analisis si aparece")
    regla("B1.3 MHSurvey", f"R: {col} fuera de rango [1,5]",
          ~df[col].between(1, 5), n, accion="Revisar codificacion si >0")
regla("B1.3 MHSurvey", "R: cgpa en formato texto-rango (no numerico)",
      pd.Series([True] * n) if df["cgpa"].dtype == object else pd.Series([False] * n),
      n, umbral_aviso=100.0,
      accion="Recodificar a ordinal antes del modelado (Objetivo 2)")

print("\n--- B1.4 Student Habits (abhi88377) ---")
df = pd.read_csv(os.path.join(BASE, "Bloque 1",
    "1 Student-Habits-vs-Academic-Performance", "student_habits_performance.csv"))
n = len(df)
regla("B1.4 StudentHabits", "R1: parental_education_level nulo",
      df["parental_education_level"].isna(), n, umbral_aviso=10.0,
      accion="Imputar por moda o excluir columna en el modelado")
regla("B1.4 StudentHabits", "R2: exam_score fuera de rango [0,100]",
      ~df["exam_score"].between(0, 100), n, accion="Recortar a [0,100] si >0")
regla("B1.4 StudentHabits", "R3: suma de horas diarias imposible (>24h)",
      (df["study_hours_per_day"] + df["social_media_hours"]
       + df["netflix_hours"] + df["sleep_hours"]) > 24, n,
      umbral_aviso=1.0, accion="Marcar como inconsistencia de captura")
regla("B1.4 StudentHabits", "R4: student_id duplicado",
      df["student_id"].duplicated(), n, accion="Debe ser clave unica (PK)")


--- B1.1 Academic Stress (poushal02) ---
  [OK   ] R1: columnas sin espacios sobrantes en el nombre: 0/9 (0.00%) -> strip() aplicado a los nombres al cargar
  [OK   ] R2: indice de estres nulo: 0/140 (0.00%) -> Excluir registro del analisis del target
  [OK   ] R3: indice de estres fuera de rango [1,5]: 0/140 (0.00%) -> Ninguna necesaria si 0
  [OK   ] R4: registros duplicados completos: 0/140 (0.00%) -> Eliminar duplicados si aparecen

--- B1.2 Anxiety GAD-7 (petalme) ---
  [OK   ] R1: GAD_T nulo: 0/13464 (0.00%) -> Excluir del analisis de ansiedad
  [OK   ] R2: GAD_T fuera de rango [0,21]: 0/13464 (0.00%) -> Revisar calculo de la escala si >0
  [OK   ] R3: Edad fuera de rango plausible [13,90]: 0/13464 (0.00%) -> Filtrar registros fuera de rango
  [AVISO] R4: Horas de juego semanales imposibles (>112): 10/13434 (0.07%) -> Tratar como atipico/captura erronea (winsorizar o excluir)
  [AVISO] R5: Work nulo (impide filtrar estudiantes): 38/13464 (0.28%) -> Los nulos quedan fuera del sub

## BLOQUE 2

**Reglas del bloque.** Carga cada dataset y ejecuta sus validaciones: nulos, rangos válidos, dominios de categorías permitidas, duplicados/claves únicas. Cada `regla()` imprime su resultado al momento.

In [3]:
print("\n--- B2.1 Social Media/Sleep/Stress (jayjoshi37) ---")
df = pd.read_csv(os.path.join(BASE, "Bloque 2",
    "2 Social Media Usage vs Sleep, Stress & Productivity",
    "social_media_sleep_stress_productivity_11000.csv"))
n = len(df)
regla("B2.1 SocialMedia", "R1: user_id duplicado", df["user_id"].duplicated(), n,
      accion="Debe ser clave unica (PK)")
regla("B2.1 SocialMedia", "R2: stress_level fuera del dominio {Low,Medium,High}",
      ~df["stress_level"].isin(["Low", "Medium", "High"]), n,
      accion="Revisar codificacion si >0")
regla("B2.1 SocialMedia", "R3: horas de pantalla fuera de rango [0,24]",
      ~df["daily_screen_time_hours"].between(0, 24), n,
      accion="Recortar o excluir si >0")
regla("B2.1 SocialMedia", "R4: sleep_hours fuera de rango [0,24]",
      ~df["sleep_hours"].between(0, 24), n, accion="Recortar o excluir si >0")

print("\n--- B2.2 SMMH (souvikahmed071) ---")
df = pd.read_csv(os.path.join(BASE, "Bloque 2", "2 Social Media and Mental Health",
                              "smmh.csv"))
n = len(df)
cols = list(df.columns)
likert = [cols[i] for i in [9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20]]
for c in likert[:3]:   # se validan todas; se listan 3 para brevedad del informe
    vals = pd.to_numeric(df[c], errors="coerce")
    regla("B2.2 SMMH", f"R: '{c[:40]}...' fuera de escala Likert [1,5]",
          ~vals.between(1, 5), n, accion="Revisar codificacion si >0")
regla("B2.2 SMMH", "R: edad no numerica o nula",
      pd.to_numeric(df[cols[1]], errors="coerce").isna(), n,
      umbral_aviso=1.0, accion="Excluir de analisis por edad")

print("\n--- B2.3 Mental Health Lifestyle (datafit17) ---")
df = pd.read_csv(os.path.join(BASE, "Bloque 2", "2 Mental_Health_Lifestyle GITHUB",
                              "Mental_Health_Lifestyle_Dataset.csv"))
n = len(df)
regla("B2.3 Lifestyle", "R1: Mental Health Condition nulo (NaN semantico)",
      df["Mental Health Condition"].isna(), n, umbral_aviso=25.0,
      accion="Recodificar NaN -> 'Ninguna' (ausencia de diagnostico)")
regla("B2.3 Lifestyle", "R2: Stress Level fuera del dominio {Low,Moderate,High}",
      ~df["Stress Level"].isin(["Low", "Moderate", "High"]), n,
      accion="Revisar codificacion si >0")
regla("B2.3 Lifestyle", "R3: Sleep Hours fuera de rango [0,24]",
      ~df["Sleep Hours"].between(0, 24), n, accion="Recortar o excluir si >0")


--- B2.1 Social Media/Sleep/Stress (jayjoshi37) ---
  [OK   ] R1: user_id duplicado: 0/11000 (0.00%) -> Debe ser clave unica (PK)
  [OK   ] R2: stress_level fuera del dominio {Low,Medium,High}: 0/11000 (0.00%) -> Revisar codificacion si >0
  [OK   ] R3: horas de pantalla fuera de rango [0,24]: 0/11000 (0.00%) -> Recortar o excluir si >0
  [OK   ] R4: sleep_hours fuera de rango [0,24]: 0/11000 (0.00%) -> Recortar o excluir si >0

--- B2.2 SMMH (souvikahmed071) ---
  [OK   ] R: '9. How often do you find yourself using ...' fuera de escala Likert [1,5]: 0/481 (0.00%) -> Revisar codificacion si >0
  [OK   ] R: '10. How often do you get distracted by S...' fuera de escala Likert [1,5]: 0/481 (0.00%) -> Revisar codificacion si >0
  [OK   ] R: '11. Do you feel restless if you haven't ...' fuera de escala Likert [1,5]: 0/481 (0.00%) -> Revisar codificacion si >0
  [OK   ] R: edad no numerica o nula: 0/481 (0.00%) -> Excluir de analisis por edad

--- B2.3 Mental Health Lifestyle (datafit17) --

## BLOQUE 3

**Reglas del bloque.** Carga cada dataset y ejecuta sus validaciones: nulos, rangos válidos, dominios de categorías permitidas, duplicados/claves únicas, la consistencia de esquema entre train y test, artefactos de corrupción de Excel en el texto ('#NAME?'). Cada `regla()` imprime su resultado al momento.

In [4]:
print("\n--- B3.1 Dreaddit train+test (payaldhokane) ---")
tr = pd.read_csv(os.path.join(BASE, "Bloque 3",
    "3 Stress Analysis in Social Media Dataset", "dreaddit-train.csv"))
te = pd.read_csv(os.path.join(BASE, "Bloque 3",
    "3 Stress Analysis in Social Media Dataset", "dreaddit-test.csv"))
regla("B3.1 Dreaddit", "R1: texto vacio o nulo (train)",
      tr["text"].isna() | (tr["text"].str.strip().str.len() == 0), len(tr),
      accion="Excluir del corpus NLP")
regla("B3.1 Dreaddit", "R2: label fuera del dominio {0,1} (train)",
      ~tr["label"].isin([0, 1]), len(tr), accion="Revisar etiquetado si >0")
regla("B3.1 Dreaddit", "R3: id duplicado (train)", tr["id"].duplicated(), len(tr),
      accion="Debe ser clave unica (PK)")
regla("B3.1 Dreaddit", "R4: solapamiento de ids entre train y test",
      te["id"].isin(set(tr["id"])), len(te),
      accion="Garantiza evaluacion sin fuga de datos (data leakage)")
regla("B3.1 Dreaddit", "R5: train y test con distinto CONJUNTO de columnas",
      pd.Series([set(tr.columns) != set(te.columns)]), 1,
      accion="Ambas particiones deben compartir esquema")
regla("B3.1 Dreaddit", "R6: train y test con distinto ORDEN de columnas",
      pd.Series([list(tr.columns) != list(te.columns)]), 1, umbral_aviso=100.0,
      accion="Mismo conjunto, distinto orden: alinear por nombre (pandas) o "
             "reordenar test con tr.columns antes de modelar")
excel_artifacts = ("#NAME?", "#REF!", "#VALUE!", "#DIV/0!", "#N/A")
regla("B3.1 Dreaddit", "R7: texto = artefacto de Excel ('#NAME?', etc.) en train",
      tr["text"].str.strip().isin(excel_artifacts), len(tr), umbral_aviso=0.5,
      accion="Excluir del corpus NLP: son celdas corruptas de la fuente original")
regla("B3.1 Dreaddit", "R8: texto = artefacto de Excel en test",
      te["text"].str.strip().isin(excel_artifacts), len(te), umbral_aviso=0.5,
      accion="Excluir del corpus NLP")

print("\n--- B3.2 Depression Student (prajwalg26) ---")
df = pd.read_csv(os.path.join(BASE, "Bloque 3",
    "3 Student_Mental_Health_Analysis GITHUB", "Depression+Student+Dataset.csv"))
n = len(df)
regla("B3.2 DepressionStudent", "R1: Depression fuera del dominio {Yes,No}",
      ~df["Depression"].isin(["Yes", "No"]), n, accion="Revisar codificacion si >0")
regla("B3.2 DepressionStudent", "R2: Academic Pressure fuera de rango [1,5]",
      ~df["Academic Pressure"].between(1, 5), n, accion="Revisar escala si >0")
regla("B3.2 DepressionStudent", "R3: Edad fuera de rango estudiantil [15,60]",
      ~df["Age"].between(15, 60), n, umbral_aviso=1.0,
      accion="Revisar registros fuera de rango")
regla("B3.2 DepressionStudent", "R4: registros duplicados completos",
      df.duplicated(), n, accion="Eliminar si aparecen")


--- B3.1 Dreaddit train+test (payaldhokane) ---
  [OK   ] R1: texto vacio o nulo (train): 0/2838 (0.00%) -> Excluir del corpus NLP
  [OK   ] R2: label fuera del dominio {0,1} (train): 0/2838 (0.00%) -> Revisar etiquetado si >0
  [OK   ] R3: id duplicado (train): 0/2838 (0.00%) -> Debe ser clave unica (PK)
  [OK   ] R4: solapamiento de ids entre train y test: 0/715 (0.00%) -> Garantiza evaluacion sin fuga de datos (data leakage)
  [OK   ] R5: train y test con distinto CONJUNTO de columnas: 0/1 (0.00%) -> Ambas particiones deben compartir esquema
  [AVISO] R6: train y test con distinto ORDEN de columnas: 1/1 (100.00%) -> Mismo conjunto, distinto orden: alinear por nombre (pandas) o reordenar test con tr.columns antes de modelar
  [AVISO] R7: texto = artefacto de Excel ('#NAME?', etc.) en train: 4/2838 (0.14%) -> Excluir del corpus NLP: son celdas corruptas de la fuente original
  [OK   ] R8: texto = artefacto de Excel en test: 0/715 (0.00%) -> Excluir del corpus NLP

--- B3.2 Depression

## BLOQUE 4

**Reglas del bloque.** Carga cada dataset y ejecuta sus validaciones: nulos, rangos válidos, dominios de categorías permitidas, el control de validez del propio instrumento (palabras inexistentes VCL6/9/12). Cada `regla()` imprime su resultado al momento.

In [5]:
print("\n--- B4.1 DASS-42 (lucasgreenwell) ---")
df = pd.read_csv(os.path.join(BASE, "Bloque 4",
    "4 Depression Anxiety Stress Scales Responses", "data.csv"), sep="\t",
    low_memory=False)
n = len(df)
regla("B4.1 DASS", "R1: edad fuera de rango plausible [13,90]",
      ~df["age"].between(13, 90), n, umbral_aviso=0.1,
      accion="Filtrar (valores tipo '1998' son anos de nacimiento mal capturados)")
item_cols = [f"Q{i}A" for i in range(1, 43)]
regla("B4.1 DASS", "R2: items DASS fuera de escala [1,4] (42 columnas)",
      (~df[item_cols].apply(lambda s: s.between(1, 4))).any(axis=1), n,
      accion="Revisar registro completo si >0")
# Control de validez del propio instrumento: VCL6, VCL9, VCL12 son palabras falsas
fake_words = df[["VCL6", "VCL9", "VCL12"]].sum(axis=1) > 0
regla("B4.1 DASS", "R3: marca palabras inexistentes (VCL6/9/12) - validez dudosa",
      fake_words, n, umbral_aviso=15.0,
      accion="Opcional: excluir o sensibilizar el analisis (control del codebook)")
regla("B4.1 DASS", "R4: genero fuera del dominio {1,2,3}",
      ~df["gender"].isin([1, 2, 3]), n, umbral_aviso=1.0,
      accion="Tratar como no informado")

print("\n--- B4.2 Mental Health dataset (chanchal24) ---")
df = pd.read_csv(os.path.join(BASE, "Bloque 4", "4 Mental Health Dataset",
                              "Mental Health dataset1.csv"))
n = len(df)
regla("B4.2 MentalHealth", "R1: columna duplicada SocialWeakness.1 == SocialWeakness",
      pd.Series([not (df["SocialWeakness"] == df["SocialWeakness.1"]).all()]), 1,
      accion="Eliminar SocialWeakness.1 (duplicado exacto verificado)")
regla("B4.2 MentalHealth", "R2: SelfEmployed nulo", df["SelfEmployed"].isna(), n,
      umbral_aviso=2.0, accion="Imputar 'No' o categoria 'Desconocido'")
regla("B4.2 MentalHealth", "R3: Treatment fuera del dominio {Yes,No}",
      ~df["Treatment"].isin(["Yes", "No"]), n, accion="Revisar si >0")
regla("B4.2 MentalHealth", "R4: Occupation fuera del dominio esperado",
      ~df["Occupation"].isin(["Student", "Corporate", "Business",
                              "Housewife", "Others"]), n,
      accion="Revisar categorias nuevas si >0")


--- B4.1 DASS-42 (lucasgreenwell) ---
  [AVISO] R1: edad fuera de rango plausible [13,90]: 8/39775 (0.02%) -> Filtrar (valores tipo '1998' son anos de nacimiento mal capturados)
  [OK   ] R2: items DASS fuera de escala [1,4] (42 columnas): 0/39775 (0.00%) -> Revisar registro completo si >0
  [AVISO] R3: marca palabras inexistentes (VCL6/9/12) - validez dudosa: 5192/39775 (13.05%) -> Opcional: excluir o sensibilizar el analisis (control del codebook)
  [AVISO] R4: genero fuera del dominio {1,2,3}: 67/39775 (0.17%) -> Tratar como no informado

--- B4.2 Mental Health dataset (chanchal24) ---
  [OK   ] R1: columna duplicada SocialWeakness.1 == SocialWeakness: 0/1 (0.00%) -> Eliminar SocialWeakness.1 (duplicado exacto verificado)
  [AVISO] R2: SelfEmployed nulo: 4356/261328 (1.67%) -> Imputar 'No' o categoria 'Desconocido'
  [OK   ] R3: Treatment fuera del dominio {Yes,No}: 0/261328 (0.00%) -> Revisar si >0
  [OK   ] R4: Occupation fuera del dominio esperado: 0/261328 (0.00%) -> Revisar ca

## RESUMEN GLOBAL

**Cuadro resumen y exportación.** Convierte todas las reglas ejecutadas en un DataFrame, lo imprime como tabla, totaliza por estado (OK/AVISO/ERROR) y lo exporta a CSV (`encoding='utf-8-sig'` para que Excel muestre bien las tildes).

In [6]:
print("\n" + "=" * 90)
print("CUADRO RESUMEN DE REGLAS DE CALIDAD")
print("=" * 90)
res = pd.DataFrame(resultados)
print(res.to_string(index=False, max_colwidth=48))

ok = (res["Estado"] == "OK").sum()
av = (res["Estado"] == "AVISO").sum()
er = (res["Estado"] == "ERROR").sum()
print(f"\nTOTAL: {len(res)} reglas | OK: {ok} | AVISO: {av} | ERROR: {er}")

res.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("Resultado exportado a:", OUT_CSV)


CUADRO RESUMEN DE REGLAS DE CALIDAD
               Dataset                                            Regla  Evaluados  Incumplen  % incumplen Estado                                           Accion
   B1.1 AcademicStress R1: columnas sin espacios sobrantes en el nombre          9          0         0.00     OK         strip() aplicado a los nombres al cargar
   B1.1 AcademicStress                        R2: indice de estres nulo        140          0         0.00     OK         Excluir registro del analisis del target
   B1.1 AcademicStress        R3: indice de estres fuera de rango [1,5]        140          0         0.00     OK                           Ninguna necesaria si 0
   B1.1 AcademicStress               R4: registros duplicados completos        140          0         0.00     OK                  Eliminar duplicados si aparecen
          B1.2 Anxiety                                   R1: GAD_T nulo      13464          0         0.00     OK                 Excluir del analis